# Pixel Traffic Unity 0.11.0 APK를 내 Google Drive에 저장

PC 설치 없이 Google 서버에서 원본 APK를 받아 본인의 Drive로 복사합니다. 계정 선택과 Drive 접근 동의는 Google 화면에서 직접 합니다.

1. 아래 코드 셀 왼쪽 **▶**를 누릅니다. 외부 노트북 실행 확인이 표시되면 코드를 확인한 뒤 진행합니다.
2. Drive 연결 화면에서 **200GB를 사용하는 Google 계정**을 선택합니다.
3. 마지막 **완료** 메시지가 표시되면 Drive 앱의 **내 Drive → Pixel Traffic APK** 폴더에서 APK를 받습니다.

코드는 고정된 공개 원본 APK 한 개를 다운로드하고 전체 SHA-256/크기/ZIP 데이터를 확인한 뒤 지정 폴더에 복사합니다. 동일 이름의 다른 파일을 덮어쓰지 않습니다. Drive를 연결하면 이 노트북이 Drive 파일에 접근할 권한을 갖습니다. 본 코드는 지정 폴더의 해당 APK만 처리합니다. 로그인 정보나 Drive 파일을 개발자에게 전송하지 않습니다.

원본: Unity 0.11.0/code12, 29,788,679bytes. 파일 SHA-256: `528363f609f6a67c59583e5e16190befe1814d71fc0dce009b9b9987afab1084`.

참고: [Google Colab 공식 안내](https://research.google.com/colaboratory/faq.html). 실제 본인 Drive 저장은 Google 접근 동의 후 실행되며, 이 노트북 준비 자체가 업로드 완료를 의미하지 않습니다.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import tempfile
import urllib.request
import zipfile

APK_NAME = "pixel-traffic-unity-prototype-0.11.0.apk"
APK_URL = "https://raw.githubusercontent.com/BACKHYUN96/S20-PLUS/ff54aa438e6f782b727b566e619b8d826bd985fc/" + APK_NAME
APK_BYTES = 29788679
APK_SHA256 = "528363f609f6a67c59583e5e16190befe1814d71fc0dce009b9b9987afab1084"


def verify_apk(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size != APK_BYTES:
        raise RuntimeError("APK 파일 크기가 검증된 원본과 다릅니다.")
    digest = hashlib.sha256()
    with path.open("rb") as reader:
        for chunk in iter(lambda: reader.read(1024 * 1024), b""):
            digest.update(chunk)
    if digest.hexdigest() != APK_SHA256:
        raise RuntimeError("APK SHA-256이 검증된 원본과 다릅니다.")


def copy_verified_apk(source, folder):
    verify_apk(source)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    destination = folder / APK_NAME
    if destination.exists():
        verify_apk(destination)
        return destination
    created = False
    try:
        with destination.open("xb") as writer:
            created = True
            with Path(source).open("rb") as reader:
                shutil.copyfileobj(reader, writer, 1024 * 1024)
        verify_apk(destination)
        return destination
    except Exception:
        if created:
            destination.unlink(missing_ok=True)
        raise


def save_to_my_drive():
    # Verify the immutable public APK before requesting Drive access.
    with tempfile.TemporaryDirectory(prefix="pixel-traffic-apk-") as temporary:
        source = Path(temporary) / APK_NAME
        print("1/3: 검증된 Unity 0.11.0 APK를 Google 서버로 받는 중…")
        with urllib.request.urlopen(APK_URL, timeout=60) as response:
            if response.status != 200:
                raise RuntimeError("APK 원본 다운로드에 실패했습니다.")
            with source.open("wb") as writer:
                shutil.copyfileobj(response, writer, 1024 * 1024)
        verify_apk(source)
        with zipfile.ZipFile(source) as archive:
            if archive.testzip() is not None or "lib/arm64-v8a/libil2cpp.so" not in archive.namelist():
                raise RuntimeError("APK 내부 데이터 확인에 실패했습니다.")
        print("원본 크기·SHA-256·APK 데이터 확인 완료.")
        print("2/3: 200GB를 사용하는 Google 계정을 선택하고 Drive 접근을 허용해주세요.")
        from google.colab import drive
        drive.mount("/content/drive")
        my_drive = Path("/content/drive/MyDrive")
        if not my_drive.is_dir():
            raise RuntimeError("내 Drive 연결을 확인할 수 없습니다.")
        destination = copy_verified_apk(source, my_drive / "Pixel Traffic APK")
        print("3/3: 원본 APK 복사 확인 완료. Drive 동기화를 요청하는 중…")
        drive.flush_and_unmount()
        print("완료: Google Drive 앱 → 내 Drive → Pixel Traffic APK → " + APK_NAME)
        print("크기: " + str(APK_BYTES) + " bytes / SHA-256: " + APK_SHA256)
        print("Drive 앱에서 파일이 보이는지 확인한 뒤 다운로드해주세요.")

save_to_my_drive()
